In [1]:
# Setup and load

import os
os.environ["OMP_NUM_THREADS"] = "14"
os.environ["MKL_NUM_THREADS"] = "14"

import scanpy as sc
from pathlib import Path

ROOT = Path("..").resolve()
sc.settings.datasetdir = ROOT / "data/raw"

raw  = sc.datasets.pbmc3k()
proc = sc.datasets.pbmc3k_processed()

print("raw :", raw.shape)
print("proc:", proc.shape)
print(proc.obs['louvain'].value_counts())

raw : (2700, 32738)
proc: (2638, 1838)
louvain
CD4 T cells          1144
CD14+ Monocytes       480
B cells               342
CD8 T cells           316
NK cells              154
FCGR3A+ Monocytes     150
Dendritic cells        37
Megakaryocytes         15
Name: count, dtype: int64


In [2]:
# Merge raw counts with labels
import numpy as np

raw.var_names_make_unique()

shared = raw.obs_names.intersection(proc.obs_names)
print("shared barcodes:", len(shared))

adata = raw[shared].copy()
adata.obs["cell_type"] = proc.obs.loc[shared, "louvain"].values

# keep the published UMAP for later plots
adata.obsm["X_umap_ref"] = proc[shared].obsm["X_umap"]

# raw counts preserved in a layer — scANVI needs this
adata.layers["counts"] = adata.X.copy()

print(adata)
print(adata.obs["cell_type"].value_counts())
print("max count value:", adata.X.max())

shared barcodes: 2638
AnnData object with n_obs × n_vars = 2638 × 32738
    obs: 'cell_type'
    var: 'gene_ids'
    obsm: 'X_umap_ref'
    layers: 'counts'
cell_type
CD4 T cells          1144
CD14+ Monocytes       480
B cells               342
CD8 T cells           316
NK cells              154
FCGR3A+ Monocytes     150
Dendritic cells        37
Megakaryocytes         15
Name: count, dtype: int64
max count value: 419.0


In [3]:
print("before:", adata.shape)

sc.pp.filter_genes(adata, min_cells=3)

print("after :", adata.shape)
print("cells unchanged:", adata.n_obs == 2638)

before: (2638, 32738)
after : (2638, 13656)
cells unchanged: True


In [4]:
# Save the processed object
from pathlib import Path

OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)

adata.write_h5ad(OUT / "pbmc3k.h5ad")

print("saved:", (OUT / "pbmc3k.h5ad").relative_to(ROOT))
print(adata)

saved: data/processed/pbmc3k.h5ad
AnnData object with n_obs × n_vars = 2638 × 13656
    obs: 'cell_type'
    var: 'gene_ids', 'n_cells'
    obsm: 'X_umap_ref'
    layers: 'counts'


In [5]:
# Export for R:

import scipy.io as sio
import scipy.sparse as sp
import pandas as pd

RDIR = ROOT / "data/processed/for_r"
RDIR.mkdir(parents=True, exist_ok=True)

X = adata.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)
mat = X.T.tocoo()

sio.mmwrite(str(RDIR / "counts.mtx"), mat)

pd.Series(adata.var_names).to_csv(RDIR / "genes.tsv", index=False, header=False)
pd.Series(adata.obs_names).to_csv(RDIR / "barcodes.tsv", index=False, header=False)
adata.obs[["cell_type"]].to_csv(RDIR / "cell_meta.csv")

print("wrote to:", RDIR.relative_to(ROOT))
print("matrix shape (genes x cells):", mat.shape)
print("saved:", (OUT / "pbmc3k.h5ad").relative_to(ROOT))

wrote to: data/processed/for_r
matrix shape (genes x cells): (13656, 2638)
saved: data/processed/pbmc3k.h5ad


In [6]:
# Verify the export
# Worth doing now rather than discovering a problem from inside R

back = sio.mmread(str(RDIR / "counts.mtx"))
g = pd.read_csv(RDIR / "genes.tsv", header=None)[0].values
b = pd.read_csv(RDIR / "barcodes.tsv", header=None)[0].values

print("mtx shape:", back.shape)
print("genes:", len(g), "barcodes:", len(b))
print("dims match:", back.shape == (len(g), len(b)))
print("values match:", np.allclose(back.tocsr()[:5,:5].toarray(),
                                   adata.X.T.tocsr()[:5,:5].toarray()))
print("first gene:", g[0], "| first barcode:", b[0])

mtx shape: (13656, 2638)
genes: 13656 barcodes: 2638
dims match: True
values match: True
first gene: AL627309.1 | first barcode: AAACATACAACCAC-1
